In [1]:
import os
import hopsworks
import pandas as pd
import joblib
from dotenv import load_dotenv
# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
#from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

In [2]:
# 1. Load your API key
load_dotenv()
api_key = os.getenv("HopsworkAPI_KEY")

project = hopsworks.login(api_key_value=api_key)

fs = project.get_feature_store()

fg = fs.get_feature_group("sialkot_aqi_features", version=1)
df = fg.read()

2026-05-31 19:09:42,356 INFO: Initializing external client
2026-05-31 19:09:42,357 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-31 19:09:47,941 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/32895
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.75s) 


In [20]:
df.head(20)

,datetime,wind_speed_100m,precipitation,apparent_temperature,wind_gusts_10m,vapour_pressure_deficit,relative_humidity_2m,aqi,co,no2,o3,so2,pm2_5,pm10,nh3,date,hour,city
0,2026-02-27 15:00:00,7.8,0.0,18.6,9.0,0.32,84,5,1161.87,34.31,42.47,10.46,106.78,177.10,57.98,2026-02-27,15,Sialkot
1,2026-02-27 16:00:00,12.6,0.0,18.6,14.4,0.46,78,5,1500.10,43.08,27.34,11.47,135.00,214.88,69.39,2026-02-27,16,Sialkot
2,2026-02-27 17:00:00,13.9,0.0,18.3,14.4,0.54,74,5,1818.34,48.04,18.38,11.01,169.68,254.94,70.48,2026-02-27,17,Sialkot
3,2026-02-27 18:00:00,6.4,0.0,19.6,17.3,0.52,76,5,2001.47,48.39,18.21,9.81,203.91,286.28,64.46,2026-02-27,18,Sialkot
4,2026-02-27 19:00:00,8.9,0.0,17.8,16.2,0.31,84,5,1876.81,38.87,33.02,8.35,221.10,291.48,51.21,2026-02-27,19,Sialkot
5,2026-02-27 20:00:00,15.0,0.0,15.8,20.5,0.27,85,5,1842.00,33.48,30.12,5.73,237.10,299.69,43.48,2026-02-27,20,Sialkot
6,2026-02-27 21:00:00,9.9,0.0,15.6,17.6,0.22,87,5,1780.16,28.58,25.98,3.98,247.55,304.75,32.34,2026-02-27,21,Sialkot
7,2026-02-27 22:00:00,6.8,0.0,14.9,3.2,0.21,87,5,1697.75,24.23,21.56,2.91,252.52,306.36,24.33,2026-02-27,22,Sialkot
8,2026-02-27 23:00:00,13.1,0.0,13.8,13.7,0.27,84,5,1657.13,21.23,16.13,2.22,259.89,312.22,21.95,2026-02-27,23,Sialkot
9,2026-02-28 00:00:00,15.4,0.0,13.3,19.4,0.33,79,5,1644.77,19.27,10.81,1.90,269.59,322.07,22.63,2026-02-28,0,Sialkot


In [4]:
print("🔪 Preparing data and splitting features...")

# We drop columns that leak the answer, or that math can't understand
columns_to_drop = ['city', 'date', 'datetime', 'aqi', 'pm2_5', 'pm10']

# TODO: Create your 'X' dataframe by dropping the columns above
X = df.drop(columns=columns_to_drop)

# TODO: Create your 'y' series using ONLY the 'pm2_5' column
y = df['pm2_5']

# TODO: Use train_test_split to hide 20% of the data. Set random_state=42 for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

🔪 Preparing data and splitting features...


In [21]:
X.head(5)

,wind_speed_100m,precipitation,apparent_temperature,wind_gusts_10m,vapour_pressure_deficit,relative_humidity_2m,co,no2,o3,so2,nh3,hour
0,7.8,0.0,18.6,9.0,0.32,84,1161.87,34.31,42.47,10.46,57.98,15
1,12.6,0.0,18.6,14.4,0.46,78,1500.10,43.08,27.34,11.47,69.39,16
2,13.9,0.0,18.3,14.4,0.54,74,1818.34,48.04,18.38,11.01,70.48,17
3,6.4,0.0,19.6,17.3,0.52,76,2001.47,48.39,18.21,9.81,64.46,18
4,8.9,0.0,17.8,16.2,0.31,84,1876.81,38.87,33.02,8.35,51.21,19


In [16]:
# Initialize the competitors
models = {
    "Ridge_Regression": Ridge(),
    "Random_Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42)
}

In [17]:
results = {}

# Train and evaluate each model in a loop
for name, model in models.items():
    # TODO: Train the model using the .fit() method on your training data
    model.fit(X_train, y_train)
    
    # TODO: Ask the model to predict PM2.5 for the hidden X_test data
    predictions = model.predict(X_test)
    
    # TODO: Calculate the three required metrics by comparing 'predictions' to 'y_test'
    rmse = root_mean_squared_error(y_test, predictions) 
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    results[name] = {"RMSE": rmse, "MAE": mae, "R2": r2, "model_object": model}
    print(f"✅ {name} trained. R2 Score: {r2:.3f}")

✅ Ridge_Regression trained. R2 Score: 0.694
2026-05-31 19:55:55,751 ERROR: Task was destroyed but it is pending!
task: <Task pending name='Task-121' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/haroon/miniconda3/envs/weatherAQI-env/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/haroon/miniconda3/envs/weatherAQI-env/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/haroon/miniconda3/envs/weatherAQI-env/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
2026-05-31 19:55:55,752 ERROR: Task was destroyed but it is pending!
task: <Task pending name='Task-122' coro=<Kernel.shell_main() running at /Users/haroon/miniconda3/envs/weatherAQI-env/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


✅ Random_Forest trained. R2 Score: 0.819
✅ XGBoost trained. R2 Score: 0.810


In [18]:
results['Ridge_Regression']

{'RMSE': 17.147792506561828,
 'MAE': 12.6642095392901,
 'R2': 0.6943782783899948,
 'model_object': Ridge()}

In [19]:
results['Random_Forest']

{'RMSE': 13.194078873060832,
 'MAE': 8.8019531598513,
 'R2': 0.819063606246251,
 'model_object': RandomForestRegressor(random_state=42)}

In [14]:
results['XGBoost']

{'RMSE': 13.533240525032303,
 'MAE': 8.703112897199327,
 'R2': 0.8096418920961704,
 'model_object': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)}

In [15]:
best_model_name = max(results, key=lambda k: results[k]['R2'])
best_metrics = results[best_model_name]
champion_model = best_metrics['model_object']

print(f"\n🏆 WINNER: {best_model_name} with an R2 of {best_metrics['R2']:.3f}")


🏆 WINNER: Random_Forest with an R2 of 0.819
